# sum-back-expand-broadcast — worked example 3: Sum Backward — verify manual gradient matches autograd

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sum-back-expand-broadcast`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A reliable way to test a manual backward implementation is to compare it to PyTorch's automatic differentiation. By calling `.sum(dim, keepdim).backward()` on a tensor with `requires_grad=True` and comparing `x.grad` to your manual `sum_back` result, you can catch both shape errors and value errors at once.

## Worked solution

**Step 1 — Set up the forward pass with autograd tracking.**
Create `x` with `requires_grad=True`. Call `out = x.sum(dim=dim, keepdim=keepdim)`. PyTorch records the operation.

**Step 2 — Create the upstream gradient.**
For simplicity use `grad_out = torch.ones_like(out)`. This simulates the loss being a plain sum of all output elements.

**Step 3 — Run autograd.**
Call `out.backward(grad_out)`. Now `x.grad` holds the reference gradient.

**Step 4 — Compute manually.**
Apply the expand-broadcast pattern: if `keepdim=False`, unsqueeze then expand; if `keepdim=True`, expand directly.

**Step 5 — Compare.**
Use `torch.allclose` to verify the two results are numerically identical. A mismatch means a shape error or a wrong unsqueeze position.

In [ ]:
import torch as t

def sum_back(grad_out: t.Tensor, x: t.Tensor, dim: int, keepdim: bool) -> t.Tensor:
    """Backward for out = x.sum(dim=dim, keepdim=keepdim)."""
    g = grad_out if keepdim else grad_out.unsqueeze(dim)
    return g.expand(x.shape)

def verify_sum_back(shape, dim, keepdim):
    t.manual_seed(7)
    x_auto = t.randn(*shape, requires_grad=True)
    out = x_auto.sum(dim=dim, keepdim=keepdim)
    grad_out = t.ones_like(out)
    out.backward(grad_out)
    ref_grad = x_auto.grad.clone()

    # Now compute manually (no requires_grad)
    x_manual = t.randn(*shape)  # same shape, values irrelevant for linear op
    our_grad = sum_back(grad_out.clone(), x_manual, dim=dim, keepdim=keepdim)

    match = t.allclose(our_grad, ref_grad)
    print(f'shape={shape}, dim={dim}, keepdim={keepdim} -> match={match}')
    return match

# Run several cases
verify_sum_back((4, 6), dim=0, keepdim=False)
verify_sum_back((4, 6), dim=1, keepdim=True)
verify_sum_back((2, 3, 5), dim=2, keepdim=False)
verify_sum_back((2, 3, 5), dim=1, keepdim=True)